# A100 1B Feasibility Probe

This notebook runs the staged MoP-Forge admission probe before any expensive 1B pilot. It detects A100 memory, selects the 40 GB or 80 GB profile, measures model/forward/backward/optimizer/eval/checkpoint phases, verifies model-only resume, and packages a lightweight report without model weights.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/NikanBHMNJ/MoP.git"
REPO = Path("/content/MoP")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,gpu]"], check=True)
subprocess.run(["mopforge", "doctor"], check=True)
subprocess.run(["mopforge", "runtime", "detect"], check=True)

In [ ]:
import json, torch

# Choose: dense, mop_full, or cached_adapter_128.
PROFILE_ROLE = "dense"
PROBE_OPTIMIZER_UPDATES = 20  # Use 20-50 updates for admission probing.

if not torch.cuda.is_available():
    raise RuntimeError("This probe requires a CUDA A100 runtime.")
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / (1024 ** 3)
if "A100" not in props.name.upper():
    raise RuntimeError(f"Expected an A100, detected {props.name}.")
memory_tier = 40 if total_gb < 60 else 80
profiles = {
    "dense": f"configs/jobs/1b_dense_a100_{memory_tier}gb_probe.json",
    "mop_full": f"configs/jobs/1b_mop_full_a100_{memory_tier}gb_probe.json",
    "cached_adapter_128": f"configs/jobs/1b_cached_adapter_128_a100_{memory_tier}gb_probe.json",
}
if PROFILE_ROLE not in profiles:
    raise ValueError(f"PROFILE_ROLE must be one of {sorted(profiles)}")
CONFIG_PATH = Path(profiles[PROFILE_ROLE])
config = json.loads(CONFIG_PATH.read_text())
if PROFILE_ROLE == "cached_adapter_128":
    payload = config["payload"]
    required = [Path(payload["activation_cache_path"]), Path(payload["base_checkpoint_path"])]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Upload/generate cached-profile prerequisites first: " + ", ".join(missing))
print({"gpu": props.name, "total_memory_gb": round(total_gb, 2), "profile": str(CONFIG_PATH)})

In [ ]:
subprocess.run(["mopforge", "gpu", "validate", str(CONFIG_PATH)], check=True)
subprocess.run(["mopforge", "gpu", "estimate", str(CONFIG_PATH)], check=True)

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT_DIR = Path("reports/a100_1b_feasibility_probe") / f"{stamp}-{PROFILE_ROLE}-{memory_tier}gb"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = REPORT_DIR / "gpu_probe_report.json"
command = [
    "mopforge", "gpu", "probe", str(CONFIG_PATH),
    "--optimizer-updates", str(PROBE_OPTIMIZER_UPDATES),
    "--output", str(REPORT_PATH),
]
result = subprocess.run(command, text=True)
if result.returncode != 0 and not REPORT_PATH.exists():
    raise RuntimeError("Probe failed before writing its diagnostic report.")
probe = json.loads(REPORT_PATH.read_text())
print(json.dumps({"status": probe["status"], "acceptance": probe["acceptance"]}, indent=2))

In [ ]:
phases = probe.get("phases", [])
for phase in phases:
    cuda = (phase.get("after") or {}).get("cuda", {})
    print(
        phase["name"],
        phase.get("status"),
        "sec=", phase.get("duration_sec"),
        "peak_reserved_gb=", cuda.get("peak_reserved_gb"),
        "peak_allocated_gb=", cuda.get("peak_allocated_gb"),
    )
print("runtime_projection=", json.dumps(probe.get("runtime_projection", {}), indent=2))
if not probe["acceptance"]["passed"]:
    print("ADMISSION BLOCKED. Inspect failed gates before a 500-update pilot.")

In [ ]:
import shutil

shutil.copy2(CONFIG_PATH, REPORT_DIR / CONFIG_PATH.name)
summary = f"""# A100 1B Feasibility Probe

- Status: `{probe['status']}`
- Admission passed: `{probe['acceptance']['passed']}`
- GPU: `{probe.get('runtime', {}).get('gpu_name')}`
- Detected memory: `{probe['acceptance'].get('detected_total_gpu_memory_gb')}` GB
- Profile: `{CONFIG_PATH}`
- Optimizer updates: `{PROBE_OPTIMIZER_UPDATES}`
- Peak reserved limit: `{probe['acceptance']['peak_reserved_limit_gb']}` GB
- Measured peak reserved: `{probe['acceptance'].get('peak_reserved_gb')}` GB

This folder contains configuration and telemetry only. Checkpoints, caches, and model weights are excluded.
"""
(REPORT_DIR / "README.md").write_text(summary, encoding="utf-8")
archive = shutil.make_archive(str(REPORT_DIR), "zip", root_dir=REPORT_DIR)
from google.colab import files
files.download(archive)